In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.autograd as autograd
import matplotlib.pyplot as plt
from pylab import rcParams
rcParams['figure.figsize'] = (5, 3)

from functools import partial
# print() helper that adds two line breaks (for readability)
printn = partial(print, end='\n\n')

seed = 1234
np.random.seed(seed)

In [2]:
# Forward pass
# 4行4列のランダムな入力データxを生成
x = torch.randn(4, 4)
# 4行1列のランダムな正解ラベルyを生成
y = torch.randn(4, 1)

# 重みを初期化
# requires_grad=True：PyTorchはこの変数に対して勾配を記録・計算するようになる
w = torch.randn(4, 1, requires_grad=True)
# バイアスの初期化
b = torch.randn(1, requires_grad=True)

# 予測値yを計算
y_pred = torch.matmul(x, w) + b

# Define the objective function
# 損失を計算
# 予測値と正解の差を取ってその合計を出す「二乗誤差」を計算
loss = (y_pred - y).pow(2).sum()

In [3]:
x = torch.randn(4, 4)
y = torch.randn(4, 1)

w = torch.randn(4, 1, requires_grad=True)
# バイアスを勾配計算する対象として宣言
b = torch.randn(1, requires_grad=True)
# b.detach()：計算グラフからbえお切り離す。これ以降、bは勾配を計算しなくてもいいただの数値のとして扱われる
b = b.detach()  # stop computing gradients for b

y_pred = torch.matmul(x, w) + b

loss = (y_pred - y).pow(2).sum()

loss.backward()

print(w.grad)  # has gradients
print(b.grad)  # has no gradients

tensor([[-3.4958],
        [-9.1797],
        [ 2.3867],
        [-4.6497]])
None


In [4]:
a = nn.Sigmoid()(torch.tensor([2]))
print(a)

a = F.sigmoid(torch.tensor([2]))
print(a)

tensor([0.8808])
tensor([0.8808])


In [5]:
# By default, a tensor does not require gradients
a = torch.zeros(1)
print(a.requires_grad)

# Wrapping with nn.Parameter makes it a learnable parameter (requires gradients)
a = nn.Parameter(a)
print(a.requires_grad)

# Turn off gradient computation for this tensor
a.requires_grad = False
print(a.requires_grad)

False
True
False


In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.autograd as autograd
import matplotlib.pyplot as plt
from pylab import rcParams
rcParams['figure.figsize'] = (5, 3)

from functools import partial
# print() helper that adds two line breaks (for readability)
printn = partial(print, end='\n\n')

seed = 1234
np.random.seed(seed)

In [ ]:
def reru(x):
    x = torch.where(x > 0, x, torch.zeros_like(x))
    return x

def softmax(x):
    x -= torch.cat([x.max(axis=1, keepdim=True).values] * x.size()[1], dim=1)
    x_exp = torch.exp(x)
    return x_exp/torch.cat([x_exp.sum(dim=1, keepdim=True)] * x.size()[1], dim=1)


class Dense(nn.Module):  # inherit from nn.Module
    def __init__(self, in_dim, out_dim, function=lambda x: x):
        super().__init__()
        # He Initialization
        # in_dim: input dimension, out_dim: output dimension
        # 重みを初期化
        # np.sqrt(6/in_dim)：Heの初期値
        # nn.Parameter：self.Wとself.bは学習させる対象（更新するパラメータ）だとPyTorchに教える
        self.W = nn.Parameter(torch.tensor(np.random.uniform(
                        low=-np.sqrt(6/in_dim),
                        high=np.sqrt(6/in_dim),
                        size=(in_dim, out_dim)
                    ).astype('float32')))
        self.b = nn.Parameter(torch.tensor(np.zeros([out_dim]).astype('float32')))
        self.function = function

    def forward(self, x):  # override forward
        return self.function(torch.matmul(x, self.W) + self.b)